# Lab 10: MLP for Time-Series Forecasting

**Course:** Machine Learning Lab  
**Topic:** Multilayer Perceptron model for AEP time-series prediction

## Lab Objective
The objective of this lab is to build and train an MLP model for time-series forecasting. The notebook prepares sequential input data, trains the model, evaluates forecasting error, and performs fine-tuning from a saved checkpoint.


## 1. Set Working Directory
The working directory is changed so the notebook can access the local dataset, helper modules, saved checkpoints, and output files.


In [1]:
import os
os.chdir(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code')

## 2. Import Libraries and Helper Functions
This cell imports regression metrics, time-series splitting utilities, plotting tools, callbacks, TensorFlow/Keras layers, and preprocessing objects required for forecasting.


In [2]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

## 3. Define Initial Parameters
The model variable, starting epoch, time steps, and number of input features are initialized before building the neural network.


In [24]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

## 4. Define the MLP Architecture
The MLP flattens the time-step input, passes it through a dense hidden layer, and produces one forecast value. This architecture treats the prepared time window as a fixed-size feature vector.


In [25]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(3 , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

## 5. Display Model Summary and Diagram
The model summary and diagram help verify the input shape, layer structure, and number of trainable parameters.


In [26]:
model1 = MLP()
model1.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_2 (Flatten)         (None, 63)                0         
                                                                 
 dense_4 (Dense)             (None, 32)                2048      
                                                                 
 dense_5 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2081 (8.13 KB)
Trainable params: 2081 (8.13 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [27]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model to work.


## 6. Set Output Paths
The checkpoint, figure, and JSON paths define where the best model and training history will be saved.


In [28]:
checkpoints = r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [29]:
os.path.exists(JSON_PATH)

True

## 7. Configure Training Callbacks
The checkpoint callback saves the best validation-loss model, while the training monitor stores loss history for later analysis.


In [30]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## 8. Compile or Load the Model
If no checkpoint is supplied, a new MLP is compiled using Adam optimizer and MAE loss. Otherwise, training can resume from an existing saved model.


In [31]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


## 9. Load Train, Validation, and Test Data
The CSV files are loaded into arrays for sequence preparation. The scaler is also loaded so predictions can later be converted back to original units.


In [32]:
import os
path_dataset =r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\sklearn\base.py:347: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## 10. Confirm MLP Function and Sequence Examples
These cells define the MLP again and test the sequence-generation function with different target lengths to understand the input-output format.


In [33]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(3, num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [34]:
univariate_multi_step(train_set, 3, target_col=0,target_len=2)

(array([[[ 0.28229128,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.27162112,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.27598902,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597]],
 
        [[ 0.27162112,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.27598902,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.23755148,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597]],
 
        [[ 0.27598902,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.23755148,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.18407588,  1.        ,  0.        , ...,  1.        ,
           0.37770796, -0.92592478]],
 
        ...,
 
        [[ 0.34731062,  1.        ,  0. 

In [35]:
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)

In [36]:
univariate_multi_step(train_set, 2, target_col=0,target_len=3)

(array([[[ 0.28229128,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.27162112,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597]],
 
        [[ 0.27162112,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.27598902,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597]],
 
        [[ 0.27598902,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597],
         [ 0.23755148,  1.        ,  0.        , ...,  1.        ,
           0.39359028, -0.91928597]],
 
        ...,
 
        [[ 0.34731062,  1.        ,  0.        , ...,  1.        ,
          -0.21352092, -0.97693849],
         [ 0.39098964,  1.        ,  0.        , ...,  1.        ,
          -0.21352092, -0.97693849]],
 
        [[ 0.39098964,  1.        ,  0.        , ...,  1.        ,
          -0.21352092, -0.97693849],
         [ 0.4376638 ,  1.        ,  

## 11. Prepare Time-Series Windows
The dataset is converted into supervised-learning samples. Each sample uses previous time steps to predict the target value.


In [37]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, 3, target_col=0,target_len=2)
validation_X, validation_y = univariate_multi_step(validation_set, 3, target_col=0,target_len=2)
test_X, test_y = univariate_multi_step(test_set, 3, target_col=0,target_len=2)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.011861085891723633 sec


In [38]:
train_X.shape

(855, 3, 21)

## 12. Train the MLP Model
The model is trained on prepared sequences and validated on a separate validation set. The callbacks save progress and the best checkpoint.


In [39]:
epochs = 2
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/2
23/27 [========================>.....] - ETA: 0s - loss: 0.2541 - mae: 0.2541 - mape: 143.9095
Epoch 1: val_loss improved from inf to 0.07205, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 3s 52ms/step - loss: 0.2347 - mae: 0.2347 - mape: 131.7270 - val_loss: 0.0720 - val_mae: 0.0720 - val_mape: 34.2880
Epoch 2/2
27/27 [==============================] - ETA: 0s - loss: 0.0998 - mae: 0.0998 - mape: 46.1851
Epoch 2: val_loss improved from 0.07205 to 0.05987, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0002-loss0.06.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 52ms/step - loss: 0.0998 - mae: 0.0998 - mape: 46.1851 - val_loss: 0.0599 - val_mae: 0.0599 - val_mape: 30.8265


## 13. Evaluate Forecasting Performance
The saved model predicts the test data. Predictions and true values are inverse-transformed, then MAE, RMSE, MAPE, and R2 are calculated to evaluate performance.


In [40]:

model = load_model(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.18.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)

MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 151ms/step
Mean Absolute Error (MAE): 12838.3
Median Absolute Error (MedAE): 12088.7
Mean Squared Error (MSE): 170821337.76
Root Mean Squared Error (RMSE): 13069.86
Mean Absolute Percentage Error (MAPE): 73.28 %
Median Absolute Percentage Error (MDAPE): 73.13 %


y_test_unscaled.shape=  (25, 2)
y_pred.shape=  (25, 1)


# Fine Tuning

## 14. Fine-Tuning Setup
Fine-tuning starts from a previous checkpoint and continues training from the selected epoch. This can improve the model if more training is needed.


In [44]:
checkpoints = r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5'
model=r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5'
start_epoch= 56

## 15. Fine-Tuning Callbacks and Model Loading
The callbacks are rebuilt and the existing model is loaded so training can continue from the saved checkpoint.


In [45]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] loading C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5...
[INFO] old learning rate: 0.0010000000474974513
[INFO] new learning rate: 9.999999747378752e-05


## 16. Continue Training
The model is trained for additional epochs using the same training and validation sequences.


In [46]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10
21/27 [======================>.......] - ETA: 0s - loss: 0.0983 - mae: 0.0983 - mape: 49.0475
Epoch 1: val_loss improved from inf to 0.06284, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 3s 59ms/step - loss: 0.0983 - mae: 0.0983 - mape: 46.5093 - val_loss: 0.0628 - val_mae: 0.0628 - val_mape: 26.8960
Epoch 2/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0959 - mae: 0.0959 - mape: 45.6956
Epoch 2: val_loss improved from 0.06284 to 0.06114, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 46ms/step - loss: 0.0933 - mae: 0.0933 - mape: 45.9270 - val_loss: 0.0611 - val_mae: 0.0611 - val_mape: 28.4452
Epoch 3/10
20/27 [=====================>........] - ETA: 0s - loss: 0.0911 - mae: 0.0911 - mape: 44.1550
Epoch 3: val_loss improved from 0.06114 to 0.05982, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5


...

27/27 [==============================] - 1s 45ms/step - loss: 0.0803 - mae: 0.0803 - mape: 39.4654 - val_loss: 0.0501 - val_mae: 0.0501 - val_mape: 25.8448
Epoch 9/10
18/27 [===================>..........] - ETA: 0s - loss: 0.0772 - mae: 0.0772 - mape: 38.6147
Epoch 9: val_loss improved from 0.05013 to 0.04856, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 48ms/step - loss: 0.0778 - mae: 0.0778 - mape: 39.4865 - val_loss: 0.0486 - val_mae: 0.0486 - val_mape: 23.3435
Epoch 10/10
19/27 [====================>.........] - ETA: 0s - loss: 0.0766 - mae: 0.0766 - mape: 37.3136
Epoch 10: val_loss improved from 0.04856 to 0.04745, saving model to C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5


c:\Users\MASTER\anaconda3\envs\ml\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


27/27 [==============================] - 1s 48ms/step - loss: 0.0756 - mae: 0.0756 - mape: 37.8537 - val_loss: 0.0474 - val_mae: 0.0474 - val_mape: 23.7455


## 17. Evaluate After Fine-Tuning
The fine-tuned checkpoint is loaded and tested again using the same evaluation metrics. This allows comparison before and after fine-tuning.


In [47]:

model = load_model(r'C:\Sohaib ur Rahman Farooqi\University\8th Semester\Machine Learning\ML Lab\Code\E1-cp-0001-loss0.07.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 [==============================] - 0s 170ms/step
Mean Absolute Error (MAE): 6086.81
Median Absolute Error (MedAE): 5705.38
Mean Squared Error (MSE): 43578498.44
Root Mean Squared Error (RMSE): 6601.4
Mean Absolute Percentage Error (MAPE): 36.04 %
Median Absolute Percentage Error (MDAPE): 32.12 %


y_test_unscaled.shape=  (25, 2)
y_pred.shape=  (25, 1)




## Conclusion
This lab demonstrates how an MLP can be used for time-series forecasting after converting sequential data into supervised-learning windows. Fine-tuning from checkpoints helps continue training and compare model performance.
